<table style="width: 100%;">
<tr>
<td style="width: 50%; text-align: right; vertical-align: middle;">
<img src="https://github.com/gitpizzanow/dummy-files/blob/main/tp3nlp.jpg?raw=true" width="150">
</td>
<td style="width: 50%; text-align: left; vertical-align: middle;">

##  (NLP)   | TF-IDF + Cosine Similarity
> [SERIE](https://tp3-nlp-ing4.netlify.app/)


* *Document Frequency (DF)*
* *IDF + Smoothing*
* *TF (Term Frequency)*
* *Cosine Similarity*



</td>
</tr>
</table>

>Data: Wikipedia-like dataset

In [56]:
from sklearn.datasets import fetch_20newsgroups

docs = fetch_20newsgroups(
    subset='train',
    remove=('headers', 'footers', 'quotes')
).data[:3000]

In [57]:
type(docs)

list

In [58]:
docs[10]

'I have a line on a Ducati 900GTS 1978 model with 17k on the clock.  Runs\nvery well, paint is the bronze/brown/orange faded out, leaks a bit of oil\nand pops out of 1st with hard accel.  The shop will fix trans and oil \nleak.  They sold the bike to the 1 and only owner.  They want $3495, and\nI am thinking more like $3K.  Any opinions out there?  Please email me.\nThanks.  It would be a nice stable mate to the Beemer.  Then I\'ll get\na jap bike and call myself Axis Motors!\n\n-- \n-----------------------------------------------------------------------\n"Tuba" (Irwin)      "I honk therefore I am"     CompuTrac-Richardson,Tx\nirwin@cmptrc.lonestar.org    DoD #0826          (R75/6)'

In [59]:

pip  install contractions


In [60]:
pip install num2words

In [61]:

pip install nltk

![STEP 1 - Preprocessing](https://img.shields.io/badge/STEP%201%20-%20Preprocessing-blue)

In [62]:
import string
import contractions
from num2words import num2words
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
#comlete the code
def preprocess(doc):
    doc = doc.lower()
    doc = contractions.fix(doc)
    words = doc.split()
    new_words = []
    for w in words:
        if w.isdigit():
            new_words.append(num2words(int(w)))
        else:
            new_words.append(w)
    doc = " ".join(new_words)
    doc = doc.translate(str.maketrans('', '', string.punctuation))
    tokens = doc.split()
    clean_tokens = []
    for t in tokens:
        if t not in stop_words:
            clean_tokens.append(lemmatizer.lemmatize(t))
    return clean_tokens


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


![STEP 2 - Vocabulary](https://img.shields.io/badge/STEP%202%20-%20Vocabulary-blue)

In [63]:
#comlete the code
def build_vocab(docs):
    vocab = set()
    for doc in docs:
        tokens = preprocess(doc)
        vocab.update(tokens)

    return sorted(list(vocab))

[![STEP 3 DF](https://img.shields.io/badge/STEP_3_DF-Document_Frequency-pink)](https://digitalpro.dev)

In [64]:
def compute_df(docs, vocab):
    df = {word: 0 for word in vocab}

    for doc in docs:
        words = set(preprocess(doc))  # unique words per doc
        for w in words:
            if w in df:
                df[w] += 1

    return df

[![STEP 4 IDF](https://img.shields.io/badge/STEP_4_IDF-Inverse_Document_Frequency-pink)](https://digitalpro.dev)

In [65]:
import math
def compute_idf(df, N):
    idf = {}

    for word, freq in df.items():
        if freq > 0:
            idf[word] = math.log(N / freq)
        else:
            idf[word] = 0

    return idf

[![STEP 5 TF Vector](https://img.shields.io/badge/STEP_5_TF-Term_Frequency_Vector-pink)](https://digitalpro.dev)

In [66]:
def compute_tf(doc, vocab):
    words = preprocess(doc)
    tf = np.zeros(len(vocab))

    for i, word in enumerate(vocab):
        tf[i] = words.count(word)

    return tf

[![STEP 6 TF-IDF Matrix](https://img.shields.io/badge/STEP_6_TFIDF-Build_TF_IDF_Matrix-pink)](https://digitalpro.dev)

In [67]:
def build_tfidf(docs):
    vocab = build_vocab(docs)
    df = compute_df(docs, vocab)
    idf = compute_idf(df, len(docs))

    tfidf_matrix = []

    for doc in docs:
        tf = compute_tf(doc, vocab)

        tfidf = np.zeros(len(vocab))
        for i, word in enumerate(vocab):
            tfidf[i] = tf[i] * idf[word]

        tfidf_matrix.append(tfidf)

    return np.array(tfidf_matrix), vocab

[![STEP 7 Cosine Similarity](https://img.shields.io/badge/STEP_7-Cosine_Similarity-pink)](https://digitalpro.dev)

In [68]:
import numpy as np
def cosine(a, b):
    dot_product = np.dot(a, b)
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    if norm_a == 0 or norm_b == 0:
        return 0
    return dot_product / (norm_a * norm_b)

[![STEP 8 Search Engine](https://img.shields.io/badge/STEP_8-Search_Engine_REAL_VERSION-orange)](https://digitalpro.dev)

In [69]:
import numpy as np
def search(query, docs, top_k=5):
    tfidf_matrix, vocab = build_tfidf(docs)

    q_vec = np.zeros(len(vocab)) # build query vector
    q_words = preprocess(query)

    for i, w in enumerate(vocab):
        q_vec[i] = q_words.count(w)

    if np.sum(q_vec) > 0:
        q_vec = q_vec / np.sum(q_vec)

    scores = []

    for i, doc_vec in enumerate(tfidf_matrix):
        score = cosine(q_vec, doc_vec)
        scores.append((score, i))

    scores.sort(reverse=True)

    return scores[:top_k]

> TEST

In [ ]:
query = "machine learning neural network"
results = search(query, docs)

for score, idx in results:
    print(score)
    print(docs[idx][:200])
    print("-" * 50)